# WeMM-Video-2B — audio-visual video understanding on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/WeMM-Video-2B-audio-visual-video-understanding.ipynb)

[`tencent/WeMM-Video-2B`](https://huggingface.co/tencent/WeMM-Video-2B) is Tencent's dense **2B** checkpoint in the WeMM-Video family (Apache-2.0, public). It is a Chinese–English bilingual omni-modal model: **text, image, audio, and video in → text out**. For video it can use the original soundtrack with the frames (`video_audio`). Hub files are **BF16 safetensors, 6.12 GiB**. Load uses `trust_remote_code` with `AutoModelForImageTextToText` + `AutoProcessor`. The Hub also reports ~3.28B parameters once the vision and Whisper-style audio encoders are counted.

**Why this model.** One small Apache-2.0 checkpoint that can describe what is on screen **and** what is said, without a separate ASR model. The [card](https://huggingface.co/tencent/WeMM-Video-2B) and [GitHub](https://github.com/Tencent/WeMM-Video) example is `examples/transformers_video.py`. Instruct mode is the default (`enable_thinking=False`); thinking mode is a card option, not this T4 smoke.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/WeMM-Video-2B-audio-visual-video-understanding.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15–16 GB of VRAM and **no native bfloat16**. Saved weights are BF16; this notebook loads **float16**. Gated-delta / RMSNorm paths in the custom code already compute in float32; if a reply is empty or NaN, LayerNorms are recast to float32 and generate is retried. GitHub install pins `transformers==5.2.0` and `torchcodec`; `config.json` records `transformers_version: 5.2.0`. **Do not reinstall torch** (Colab already has a CUDA wheel). This notebook keeps Colab's torch, installs `transformers>=5.2.0` (current Colab 5.x already satisfies that), and uses **PyAV / librosa** if `torchcodec` cannot decode. Video is capped at **8 frames / 1 fps / 262144 total pixels** so the T4 fits (the processor default is `max_frames=128` and `total_pixels=25165824` — that will OOM 15 GB).

**Not in this notebook:** patched vLLM / SGLang serving, thinking mode, GGUF / AWQ / FP8 / NVFP4 checkpoints, or the 4B / 9B / 35B-A3B family members.

**Links:** [model card](https://huggingface.co/tencent/WeMM-Video-2B) · [GitHub](https://github.com/Tencent/WeMM-Video) · [technical report](https://arxiv.org/abs/2611.26409) · [announcement](https://x.com/HuggingPapers/status/2123750396002767170) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/WeMM-Video-2B-audio-visual-video-understanding.md) · [collection](https://huggingface.co/collections/tencent/wemm-video)


## 0. Check the GPU

Confirm CUDA is available, print the device name and free VRAM. Need a T4 (or any CUDA GPU). A restart after the install cell drops this state; section 2 prints it again before load.


In [ ]:
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; load will use float16.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install (keep Colab's torch)

The [GitHub Quick Start](https://github.com/Tencent/WeMM-Video#install-dependencies) install is:

```bash
python -m pip install --index-url https://download.pytorch.org/whl/cu129 \
  torch==2.10.0 torchvision==0.25.0 torchcodec==0.10.0
python -m pip install transformers==5.2.0 accelerate pillow librosa soundfile scipy requests
```

`config.json` was saved with `transformers_version: 5.2.0`. Custom code (`modeling_wemm.py`, `processing_wemm.py`) is loaded with **`trust_remote_code=True`**. This notebook pins **`transformers>=5.2.0`** so a current Colab 5.x runtime is kept (do not downgrade to 5.2.0). `accelerate` is for `device_map="auto"`. `apply_chat_template` needs **`jinja2>=3.1.6`**. `scipy` / `requests` / `soundfile` / `librosa` are imported by the processor. **`av`** writes the fallback clip and decodes video if `torchcodec` cannot load.

**Do not reinstall torch** — Colab already ships a CUDA wheel; forcing `torch==2.10.0` from the cu129 index often drops GPU support. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled torch.

**Do not import the model in this cell.** Colab has already imported its preinstalled transformers. After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata as md
import os
import signal
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

INSTALL_TAG = "tf>=5.2.0-wemm-video-2b"
MARKER = (
    Path("/content/.wemm_video_2b_nb_install_ok")
    if Path("/content").is_dir()
    else Path(".wemm_video_2b_nb_install_ok")
)
already_restarted = MARKER.exists() and MARKER.read_text().strip() == INSTALL_TAG

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# Public weights: skip Colab userdata.get("HF_TOKEN") popups.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

# GitHub: transformers==5.2.0 accelerate pillow librosa soundfile scipy requests
# (+ torchcodec). Keep Colab's CUDA torch. Do not pin 4.x.
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers>=5.2.0" accelerate pillow librosa soundfile scipy requests av "jinja2>=3.1.6"',
)
try:
    get_ipython().run_line_magic("pip", "install -q torchcodec")
except Exception as exc:
    print("torchcodec pip skipped:", repr(exc))

print()
print("transformers", md.version("transformers"))
print("accelerate", md.version("accelerate"))
print("huggingface_hub", md.version("huggingface_hub"))
print("jinja2", md.version("jinja2"))
for pkg in ("librosa", "soundfile", "scipy", "requests", "av"):
    try:
        print(pkg, md.version(pkg))
    except md.PackageNotFoundError:
        print(pkg, "not installed")
try:
    print("torchcodec", md.version("torchcodec"))
except md.PackageNotFoundError:
    print("torchcodec not installed (PyAV / librosa fallback will be used)")

MARKER.write_text(INSTALL_TAG)
if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so the installed transformers is imported.")
    print("After reconnect: skip this cell and run from section 2 (runtime check).")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> The install cell **restarts the session for you**. After reconnect, skip it and continue from section 2.
> A "session crashed" message after install is the restart, not a failed pip. If you restarted by hand instead: *Runtime > Restart session*, then this next cell.


## 2. Runtime check after restart

Confirm `transformers>=5.2.0`, CUDA, device name, and free VRAM. T4 (`sm_75`) has no native bfloat16 — **float16** is what this notebook loads. Cap video sampling here so a later cell cannot accidentally use the processor default of 128 frames.


In [ ]:
import os
import time
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
# Processor defaults are max_frames=128 / total_pixels=25165824 (T4 OOM).
# Card example uses fps=1, min_frames=4, max_frames=16, total_pixels=262144.
os.environ["WEMM_VIDEO_FPS"] = "1"
os.environ["WEMM_VIDEO_MIN_FRAMES"] = "4"
os.environ["WEMM_VIDEO_MAX_FRAMES"] = "8"
os.environ["WEMM_VIDEO_LONGEST_EDGE"] = "262144"
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.2.0"), (
    f"transformers {transformers.__version__} is loaded; WeMM-Video-2B needs 5.2+ "
    "(config.json transformers_version 5.2.0). Run the install cell, then "
    "Runtime > Restart session."
)

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda"
# T4 (sm_75) has no native bfloat16. Checkpoint dtype is bfloat16.
DTYPE = torch.float16 if major < 8 else torch.bfloat16
MODEL_ID = "tencent/WeMM-Video-2B"
# Even frame counts: temporal_patch_size=2. Card example max_frames=16; T4 uses 8.
CLIP = dict(fps=1.0, min_frames=4, max_frames=8, total_pixels=262144)

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}  (saved weights are bfloat16; T4 has no native bf16)")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
if major < 8:
    print("No native bfloat16 on this GPU — staying on float16.")
print("video cap:", CLIP)


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB | "
        f"free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load the model

The card's call is:

```python
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "tencent/WeMM-Video-2B"
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_id, trust_remote_code=True, dtype=torch.bfloat16, device_map="auto",
).eval()
```

We only change **`dtype=torch.float16`** on T4 (the card's `bfloat16` is the wrong type on `sm_75`) and try **SDPA**, then **eager**. First run downloads **6.12 GiB** of BF16 safetensors. `trust_remote_code` is required (`auto_map` → `WeMMVideoForConditionalGeneration` / `WeMMProcessor`).

`config.json` records `dtype: bfloat16` on the text, vision, and audio configs. `h5_modeling_whisper.py` also **hardcodes** `torch.bfloat16` on the audio Q-Former (`BertLMHeadModel._from_config(..., torch_dtype=torch.bfloat16)`) and `audio_proj`. After load we recast leftover bf16 parameters/buffers to float16 (leave float32 SSM states alone). Processor floating tensors (`pixel_values`, `pixel_values_videos`, `audio_feature`) are cast to the same dtype before `generate`, otherwise `F.layer_norm` raises `expected scalar type Half but found BFloat16`.

A missing optional `liger_kernel` import in the custom modeling file may print `No module named 'liger_kernel'` during load; generate still runs without it.


In [ ]:
import io
import math
import sys

import numpy as np
from transformers import AutoModelForImageTextToText, AutoProcessor

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_wemm(attn):
    # Transformers 5.22: `dtype` is canonical; `torch_dtype` is still the alias
    # nested auto_map / `_from_config` paths read (Whisper Q-Former uses torch_dtype=).
    kwargs = dict(
        trust_remote_code=True,
        dtype=DTYPE,
        device_map="auto",
        attn_implementation=attn,
        token=False,
    )
    try:
        return AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, torch_dtype=DTYPE, **kwargs
        ).eval()
    except TypeError:
        return AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs).eval()


t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, token=False)

ATTN = None
model = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_wemm(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} {DTYPE} load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load WeMM-Video-2B") from last_error

torch.cuda.synchronize()
print(f"loaded {MODEL_ID} in {time.perf_counter() - t0:.1f}s")
print("class:", type(model).__name__)
print("attention:", ATTN, "| dtype:", next(model.parameters()).dtype)
print("device:", next(model.parameters()).device)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e9:.2f}B  (card: dense 2B; Hub reports ~3.28B with encoders)")


def _dtype_census(mod, label):
    from collections import Counter

    counts = Counter()
    for p in mod.parameters():
        counts[f"param:{p.dtype}"] += p.numel()
    for b in mod.buffers():
        if b is not None and torch.is_floating_point(b):
            counts[f"buf:{b.dtype}"] += b.numel()
    print(label, dict(counts))
    return counts


def _cast_leftover_bf16(mod, dtype):
    """h5_modeling_whisper.py hardcodes Q-Former and audio_proj as bfloat16."""
    for name, sub in mod.named_modules():
        if name.endswith(("qformer", "audio_proj")):
            try:
                sub.to(dtype=dtype)
            except Exception as exc:
                print(f"could not .to({dtype}) {name}: {exc}")
    n_param = n_buf = 0
    with torch.no_grad():
        for p in mod.parameters():
            if p.dtype == torch.bfloat16:
                p.data = p.data.to(dtype=dtype)
                n_param += 1
        for b in mod.buffers():
            if b is not None and b.is_floating_point() and b.dtype == torch.bfloat16:
                b.data = b.data.to(dtype=dtype)
                n_buf += 1

    def _fix_cfg(cfg, seen=None):
        if cfg is None:
            return
        if seen is None:
            seen = set()
        ident = id(cfg)
        if ident in seen:
            return
        seen.add(ident)
        if isinstance(cfg, dict):
            if cfg.get("dtype") in (torch.bfloat16, "bfloat16", "bf16"):
                cfg["dtype"] = dtype
            for value in cfg.values():
                if isinstance(value, (dict,)) or hasattr(value, "dtype"):
                    _fix_cfg(value, seen)
            return
        dt = getattr(cfg, "dtype", None)
        if dt in (torch.bfloat16, "bfloat16", "bf16"):
            try:
                cfg.dtype = dtype
            except Exception:
                pass
        for key in ("text_config", "vision_config", "audio_config", "qformer_config"):
            child = getattr(cfg, key, None)
            if child is not None:
                _fix_cfg(child, seen)

    _fix_cfg(getattr(mod, "config", None))
    return n_param, n_buf


_dtype_census(model, "dtypes before bf16 recast")
n_p, n_b = _cast_leftover_bf16(model, DTYPE)
print(f"recast leftover bfloat16 -> {DTYPE}: {n_p} tensors, {n_b} buffers")
_dtype_census(model, "dtypes after bf16 recast")
print_vram("after load")


def _patch_wemm_media_decoders():
    """Card decode path is torchcodec. Colab often cannot load it; keep PyAV / librosa."""
    proc_mod = sys.modules[processor.__class__.__module__]
    pkg = proc_mod.__name__.rsplit(".", 1)[0]
    media_mod = sys.modules[f"{pkg}.wemm_media"]
    policy_mod = sys.modules[f"{pkg}.wemm_video_policy"]
    orig_video = media_mod.decode_video
    orig_audio = media_mod.decode_audio

    def decode_video(source, **kwargs):
        try:
            return orig_video(source, **kwargs)
        except Exception as exc:
            print(f"torchcodec video decode failed ({type(exc).__name__}: {exc}); using PyAV.")
            return _decode_video_av(source, policy_mod, **kwargs)

    def decode_audio(source):
        try:
            return orig_audio(source)
        except Exception as exc:
            print(
                f"torchcodec audio decode failed ({type(exc).__name__}: {exc}); "
                "using soundfile/librosa."
            )
            return _decode_audio_sf(source, media_mod.resolve_media)

    media_mod.decode_video = decode_video
    media_mod.decode_audio = decode_audio
    proc_mod.decode_video = decode_video
    proc_mod.decode_audio = decode_audio
    print("media decoders: torchcodec with PyAV/librosa fallback")


def _decode_video_av(source, policy_mod, **kwargs):
    import av

    options = policy_mod.resolve_video_options(kwargs)
    container = av.open(io.BytesIO(source) if isinstance(source, bytes) else str(source))
    stream = container.streams.video[0]
    origin_fps = float(stream.average_rate) if stream.average_rate else 25.0
    frames = [frame.to_ndarray(format="rgb24") for frame in container.decode(video=0)]
    container.close()
    total = len(frames)
    if total < 2:
        raise ValueError("Video needs at least two frames")
    count = policy_mod.video_frame_count(total, origin_fps, options)
    indices = np.linspace(0, total - 1, count, dtype=int).tolist()
    sampled = np.stack([frames[i] for i in indices], axis=0)
    return sampled, dict(
        total_num_frames=total,
        fps=origin_fps,
        frames_indices=indices,
        duration=total / origin_fps,
        width=int(sampled.shape[2]),
        height=int(sampled.shape[1]),
        video_backend="av",
    )


def _decode_audio_sf(source, resolve_media):
    import soundfile as sf
    from scipy.signal import resample_poly

    if isinstance(source, dict):
        if source.get("array") is not None:
            source = (source["array"], source.get("sampling_rate", 16000))
        else:
            source = source["path"]
    source = resolve_media(source)
    if isinstance(source, tuple) and len(source) == 2:
        samples = np.asarray(source[0], dtype=np.float32)
        rate = int(source[1])
    else:
        try:
            handle = io.BytesIO(source) if isinstance(source, bytes) else source
            samples, rate = sf.read(handle, dtype="float32", always_2d=False)
        except Exception:
            import librosa

            handle = io.BytesIO(source) if isinstance(source, bytes) else source
            samples, _ = librosa.load(handle, sr=16000, mono=True)
            return np.asarray(samples, dtype=np.float32)
        samples = np.asarray(samples, dtype=np.float32)
        if samples.ndim > 1:
            samples = samples.mean(axis=-1)
        rate = int(rate)
    if rate != 16000:
        divisor = math.gcd(rate, 16000)
        samples = resample_poly(samples, 16000 // divisor, rate // divisor)
    samples = np.asarray(samples, dtype=np.float32)
    if samples.size == 0 or not np.isfinite(samples).all():
        raise ValueError("Audio must contain finite, nonempty samples")
    return samples


_patch_wemm_media_decoders()


INPUT_KEYS = (
    "input_ids",
    "attention_mask",
    "pixel_values",
    "image_grid_thw",
    "pixel_values_videos",
    "video_grid_thw",
    "audio_feature",
    "position_ids",
    "cache_position",
)


def _prepare_inputs(batch):
    """Move tensors to the GPU and cast floating processor outputs to DTYPE.

    modeling_wemm.py does pixel_values.type(self.visual.dtype) and
    audio_feature.type(self.audio.dtype). If those stay bf16 while the LM is
    float16, Whisper/BERT LayerNorm raises: expected Half but found BFloat16.
    """
    device = next(model.parameters()).device
    names = set(INPUT_KEYS)
    names.update(getattr(processor, "model_input_names", []) or [])
    prepared = {}
    for key, value in batch.items():
        if key not in names or not torch.is_tensor(value):
            continue
        if value.is_floating_point():
            prepared[key] = value.to(device=device, dtype=DTYPE)
        else:
            prepared[key] = value.to(device=device)
    return prepared


def _looks_bad(text):
    if text is None:
        return True
    stripped = str(text).strip()
    if not stripped:
        return True
    lower = stripped.lower()
    return lower in {"nan", "none"} or " nan" in lower


def ask(prompt, *media, max_new_tokens=128):
    """Card usage: apply_chat_template(..., enable_thinking=False) -> generate."""
    content = [*media, {"type": "text", "text": prompt}]
    raw = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = _prepare_inputs(raw)
    prompt_len = inputs["input_ids"].shape[1]
    try:
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    except RuntimeError as exc:
        if "Half" not in str(exc) and "BFloat16" not in str(exc) and "bfloat16" not in str(exc).lower():
            raise
        print(f"dtype mismatch ({exc}); recasting leftover bfloat16 to {DTYPE} and retrying.")
        _cast_leftover_bf16(model, DTYPE)
        inputs = _prepare_inputs(raw)
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer_ids = output[0, prompt_len:]
    text = processor.tokenizer.decode(answer_ids, skip_special_tokens=True)
    if _looks_bad(text):
        print(f"Empty/NaN reply in {DTYPE}; recasting leftover bfloat16 and retrying once.")
        _cast_leftover_bf16(model, DTYPE)
        inputs = _prepare_inputs(raw)
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        answer_ids = output[0, prompt_len:]
        text = processor.tokenizer.decode(answer_ids, skip_special_tokens=True)
    print_vram("after generate")
    return text


## 4. Sample media

Official Apollo 11 examples from [Tencent/WeMM-Video `examples/media`](https://github.com/Tencent/WeMM-Video/tree/main/examples/media) (NASA archives; see the repo's media credits). Fallbacks: NASA's Earth still, Hugging Face dummy speech, then a short synthesized clip so later cells still run.


In [ ]:
import shutil
import subprocess
import urllib.request

import av
import numpy as np
import soundfile as sf
from IPython.display import Video, display
from PIL import Image

FALLBACK_VIDEO_AUDIO = None

SAMPLE_DIR = Path("wemm_video_2b_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}

GH = "https://raw.githubusercontent.com/Tencent/WeMM-Video/main/examples/media"
NASA_EARTH = "https://images-assets.nasa.gov/image/as11-36-5355/as11-36-5355~small.jpg"
COCO_CATS = "http://images.cocodataset.org/val2017/000000039769.jpg"
HF_WAV = (
    "https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples"
    "/resolve/main/librispeech_mr_quilter.wav"
)
HF_FLAC = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"


def _urlretrieve(url, path, timeout=120):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as resp, open(path, "wb") as f:
        f.write(resp.read())
    return Path(path)


def download_first(urls_and_names):
    last_err = None
    for url, name in urls_and_names:
        dest = SAMPLE_DIR / name
        try:
            if not dest.exists() or dest.stat().st_size < 32:
                print("Downloading", url)
                _urlretrieve(url, dest)
            if dest.stat().st_size < 32:
                raise OSError(f"{dest} is too small")
            print(f"loaded {dest.name}: {dest.stat().st_size / 1024:.1f} KiB")
            return dest
        except Exception as exc:
            last_err = exc
            print(f"failed {url}: {exc!r}")
            dest.unlink(missing_ok=True)
    raise RuntimeError(f"all downloads failed: {last_err!r}")


def write_fallback_clip(path, n_frames=8, fps=2, size=320, sr=16000):
    """Tiny moving-square MP4 plus a sidecar WAV (espeak if present, else a tone).

    The processor's video_audio item can take a separate `audio` field, so we do
    not have to mux AAC (unreliable on some Colab PyAV builds).
    """
    global FALLBACK_VIDEO_AUDIO
    wav_path = SAMPLE_DIR / "fallback-voice.wav"
    spoken = "The orange square moves down the screen."
    try:
        if not shutil.which("espeak-ng"):
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True)
        subprocess.run(
            ["espeak-ng", "-s", "140", "-v", "en-us", "-w", str(wav_path), spoken],
            check=True,
        )
    except Exception as exc:
        print("espeak fallback failed, using a 440 Hz tone:", repr(exc))
        t = np.arange(int(sr * n_frames / fps), dtype=np.float32) / sr
        sf.write(wav_path, (0.2 * np.sin(2 * np.pi * 440 * t)).astype(np.float32), sr)
    FALLBACK_VIDEO_AUDIO = wav_path

    container = av.open(str(path), mode="w")
    stream = container.add_stream("mpeg4", rate=fps)
    stream.width = size
    stream.height = size
    stream.pix_fmt = "yuv420p"
    for i in range(n_frames):
        frame = np.full((size, size, 3), (20, 70, 50), dtype=np.uint8)
        y = int((i / max(n_frames - 1, 1)) * (size - 64))
        frame[y : y + 64, 128:192] = (250, 80, 40)
        vf = av.VideoFrame.from_ndarray(frame, format="rgb24")
        for packet in stream.encode(vf):
            container.mux(packet)
    for packet in stream.encode():
        container.mux(packet)
    container.close()
    return Path(path)


try:
    VIDEO_PATH = download_first(
        [
            (f"{GH}/apollo-11-first-steps.mp4", "apollo-11-first-steps.mp4"),
        ]
    )
except Exception as exc:
    print("official clip download failed:", repr(exc))
    VIDEO_PATH = write_fallback_clip(SAMPLE_DIR / "fallback-moving-square.mp4")
    print("wrote", VIDEO_PATH)

try:
    IMAGE_PATH = download_first(
        [
            (f"{GH}/apollo-11-earth.jpg", "apollo-11-earth.jpg"),
            (NASA_EARTH, "nasa-as11-36-5355.jpg"),
            (COCO_CATS, "coco-cats.jpg"),
        ]
    )
except Exception as exc:
    print("image download failed, drawing a fallback:", repr(exc))
    IMAGE_PATH = SAMPLE_DIR / "fallback-earth.png"
    Image.new("RGB", (320, 320), (20, 40, 120)).save(IMAGE_PATH)

try:
    AUDIO_PATH = download_first(
        [
            (f"{GH}/apollo-11-small-step.wav", "apollo-11-small-step.wav"),
            (HF_WAV, "librispeech_mr_quilter.wav"),
            (HF_FLAC, "narsil-asr-dummy-1.flac"),
        ]
    )
except Exception as exc:
    print("audio download failed, synthesizing:", repr(exc))
    AUDIO_PATH = SAMPLE_DIR / "fallback-speech.wav"
    try:
        if not shutil.which("espeak-ng"):
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True)
        subprocess.run(
            [
                "espeak-ng",
                "-s",
                "140",
                "-v",
                "en-us",
                "-w",
                str(AUDIO_PATH),
                "That's one small step for a man, one giant leap for mankind.",
            ],
            check=True,
        )
    except Exception as synth_exc:
        print("espeak failed, writing a tone:", repr(synth_exc))
        t = np.arange(16000 * 3, dtype=np.float32) / 16000
        sf.write(AUDIO_PATH, 0.2 * np.sin(2 * np.pi * 440 * t), 16000)

print("video:", VIDEO_PATH)
print("image:", IMAGE_PATH)
print("audio:", AUDIO_PATH)


## 5. Demo 1 — video + soundtrack (`video_audio`)

Card item type **`video_audio`**: frames **and** the original audio track. Prompt follows `examples/transformers_video.py`. Sampling matches the card example but with **`max_frames=8`** (card snippet uses 16) so a free T4 stays under 15 GB. Raise `max_frames` on a larger GPU.


In [ ]:
display(Video(str(VIDEO_PATH), embed=True, width=320))

clip = dict(video=str(VIDEO_PATH), **CLIP)
if FALLBACK_VIDEO_AUDIO is not None:
    clip["audio"] = str(FALLBACK_VIDEO_AUDIO)
    print("using sidecar audio for the synthesized clip:", FALLBACK_VIDEO_AUDIO)
print("clip options:", {k: clip[k] for k in clip if k != "video"})
print(
    ask(
        "Describe the scene and summarize the speech.",
        {"type": "video_audio", **clip},
    )
)


## 6. Demo 2 — image question

Card example: the Apollo 11 Earth photograph (`examples/media/apollo-11-earth.jpg`). Fallback is NASA's published still, then the COCO cats photo used in the other vision notebooks.


In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
display(image.resize((320, 320)))

print(
    ask(
        "Which planet is shown? Describe what you see.",
        {"type": "image", "image": str(IMAGE_PATH)},
    )
)


## 7. Demo 3 — audio transcription / summary

Card example: transcribe the English speech in the Apollo 11 “small step” recording. If that file did not download, this cell uses the Hugging Face dummy clip or the synthesized fallback.


In [ ]:
from IPython.display import Audio, display

display(Audio(str(AUDIO_PATH)))

print(
    ask(
        "Transcribe the English speech in this recording. Then give a one-sentence summary.",
        {"type": "audio", "audio": str(AUDIO_PATH)},
    )
)


## 8. Caveats and cleanup

**What this notebook is for.** The card's Transformers path on a free T4: `video_audio` (frames + soundtrack), an image question, and an audio transcription/summary. `torch.cuda.max_memory_allocated` is printed after load, after each generate, and again below.

**What it is not.** Patched vLLM / SGLang serving (GitHub requires a runtime patch), thinking mode, GGUF / AWQ / FP8 / NVFP4, or the 4B / 9B / 35B-A3B checkpoints. The [35B-A3B evaluation table](https://huggingface.co/tencent/WeMM-Video-35BA3B#evaluation) is not a 2B result and is not rerun here.

**T4 / dtype.** Float16 + SDPA. The card snippet uses `dtype=torch.bfloat16`. A T4 has no native bf16. `h5_modeling_whisper.py` hardcodes the audio Q-Former and `audio_proj` as `torch.bfloat16`; the load cell recasts leftover bf16 parameters/buffers to float16 (float32 SSM states stay float32). Processor floating tensors are cast to float16 before generate. Reloading the whole model in float32 (~12 GB of weights) will likely OOM a 15 GB card.

**Video budget.** Processor default `max_frames=128` / `total_pixels=25165824` will OOM. This notebook uses the card example's `total_pixels=262144` and **8** frames. Frame counts stay even (`temporal_patch_size=2`).

**Decode path.** Custom `wemm_media.py` uses `torchcodec`. If that import or decode fails, the load cell patches in PyAV (video) and soundfile/librosa (audio). Skip reinstalling Colab's torch to get a matching torchcodec wheel.

**Related notebooks:** [LFM3-Omni-2B](LFM3-Omni-2B.ipynb) (image / ASR / short video, text out), [LFM3-VL-2B](LFM3-VL-2B-vision.ipynb), [LFM3-Audio-2B](LFM3-Audio-2B-voice.ipynb), [WeVisOne-VL 2B](WeVisOne-VL-2B-detect-segment.ipynb) (boxes / cutouts, not omni audio-visual), [MiniCPM-V 4.7 1B](MiniCPM-V-4.7-1B-vision.ipynb).

Free the GPU before you open another notebook:


In [ ]:
import gc

print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del model, processor
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
